# H2 com o prompt revisado (v3) — rodar durante a noite (Colab gratuito, GPU T4)

**Antes de rodar:** Ambiente de execução → Alterar o tipo → **GPU T4**. Depois: **Ambiente de execução → Executar tudo**. Deixe a aba aberta e o Mac ligado na tomada.

O que muda em relação à primeira rodada da H2: só o prompt. O v3 acrescenta regras de desempate escritas a partir dos erros do v2 na validação (o Qwen marcava "outros" e "anúncio" demais e "recebimento" de menos). Modelo, exemplos, decodificação e treino do BERTimbau são os mesmos.

1. Qwen2.5-7B com o prompt v3 rotula a validação (599) e o treino (5.589) — ~2,5 h
2. BERTimbau treinado com esses rótulos (3 sementes), avaliado no mesmo teste humano — ~10 min
3. Tudo fica no Drive em `MyDrive/pln-absa-olist/resultados_v3/`; no fim baixa `resultados_v3.zip`. Se cair, abra de novo e Executar tudo: continua de onde parou.

Atenção na leitura: as regras foram escritas olhando a validação, então a concordância na validação é otimista. A medida limpa é o teste.

In [ ]:
# 1) Repositório, pacote e Drive
import os, sys
from pathlib import Path
if "google.colab" in sys.modules:
    !git clone -q https://github.com/vnevesl/pln-absa-olist.git 2>/dev/null || git -C pln-absa-olist pull -q
    %cd /content/pln-absa-olist
    !pip -q install -e ".[llm]"
    from google.colab import drive
    try:
        drive.mount("/content/drive")
        BASE = Path("/content/drive/MyDrive/pln-absa-olist")
    except Exception as erro:
        print(f"⚠️ Drive não conectou ({erro}). NÃO feche a aba até baixar o zip no fim.")
        BASE = Path("/content/pln-absa-olist-saida")
else:
    BASE = Path("results/colab_local")
_src = str(Path("src").resolve())
if _src not in sys.path:
    sys.path.insert(0, _src)
SAIDA = BASE / "resultados_v3"; SAIDA.mkdir(parents=True, exist_ok=True)
CACHE = BASE / "llm_cache"; CACHE.mkdir(parents=True, exist_ok=True)
CKPT = Path("/content/ckpt" if "google.colab" in sys.modules else BASE / "ckpt"); CKPT.mkdir(parents=True, exist_ok=True)
Path("data").mkdir(exist_ok=True)
if not Path("data/llm_cache").exists():
    os.symlink(CACHE, "data/llm_cache")
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NENHUMA — troque o ambiente para GPU T4!")

In [ ]:
# 2) Dados (mesma divisão da primeira rodada)
import gc, json, time
import pandas as pd
from sklearn.model_selection import train_test_split
from absa.config import ASPECTS, LABEL_COLUMNS, POLARITIES, SEED
from absa.llm_labeler import hf_generator, label_texts, select_fixed_examples
from absa.metrics import bootstrap_ci, macro_f1, per_aspect_report, tuple_f1
from absa.models.encoder import ENCODERS, EncoderAspectClassifier, TrainConfig
from absa.preprocess import clean_light
from absa.repro import load_repro

treino, val, teste, _ = load_repro()
demo, val_dev = train_test_split(val, train_size=200, stratify=val.nota, random_state=SEED)
demo, val_dev = demo.reset_index(drop=True), val_dev.reset_index(drop=True)
exemplos = select_fixed_examples(demo, k=6)
entrada = lambda df: df.texto_modelo.map(clean_light).tolist()
MODELO, VERSAO = "Qwen/Qwen2.5-7B-Instruct", "v3"
print(f"treino {len(treino)} · val_dev {len(val_dev)} · prompt {VERSAO}")

In [ ]:
# 3) Qwen com o prompt v3: validação (599) e treino (5.589). Parte longa; retomável pelo cache no Drive
ja_rotulado = all((SAIDA / f"repro_{n}_qwen7b_v3.csv").exists() for n in ["valdev", "treino"])
generate = None if ja_rotulado else hf_generator(MODELO, load_in_4bit=True, max_new_tokens=60)

def rotular(df, nome, lote=16):
    arq = SAIDA / f"repro_{nome}_qwen7b_v3.csv"
    if arq.exists():
        print(f"✓ {nome} já rotulado — lendo do Drive"); return pd.read_csv(arq)
    t0 = time.time()
    rot, _ = label_texts(df.texto_modelo.tolist(), generate, MODELO, exemplos,
                         cache_name=f"qwen7b_v3_{nome}", batch_size=lote, version=VERSAO)
    out = df[["review_id"]].copy()
    for a in ASPECTS:
        out[f"asp_{a}"] = [r[a] if r else None for r in rot]
    out["valido"] = [r is not None for r in rot]
    out.to_csv(arq, index=False)
    print(f"{nome}: {out.valido.sum()}/{len(out)} respostas válidas ({(time.time() - t0) / 60:.1f} min)")
    return out

def sem_invalidos(rot):
    pred = rot[LABEL_COLUMNS].copy()
    pred.loc[~rot.valido.to_numpy(), LABEL_COLUMNS] = "ausente"
    return pred.fillna("ausente")

def alinhar(rot, ref):
    return rot.set_index("review_id").loc[ref.review_id].reset_index()

rot_val = alinhar(rotular(val_dev, "valdev"), val_dev)
f1, lo, hi = bootstrap_ci(val_dev, sem_invalidos(rot_val))
n_asp = (sem_invalidos(rot_val) != "ausente").sum(axis=1).mean()
print(f"Concordância v3 na validação: F1 de pares {f1:.3f} (IC95% {lo:.3f}–{hi:.3f}) · {n_asp:.2f} aspectos por avaliação"
      " (v2: 0,772 e 1,96; humanos: 1,54)")
display(per_aspect_report(val_dev, sem_invalidos(rot_val)).round(3))

rot_treino = alinhar(rotular(treino, "treino"), treino)
arq_qual = SAIDA / "qualidade_rotulos_qwen7b_v3.json"
if not arq_qual.exists():
    json.dump({"versao": VERSAO, "f1_pares_valdev": f1, "ic_valdev": [lo, hi], "aspectos_por_avaliacao_valdev": n_asp,
               "f1_pares_treino": tuple_f1(treino, sem_invalidos(rot_treino)),
               "validos_valdev": int(rot_val.valido.sum()), "validos_treino": int(rot_treino.valido.sum())},
              open(arq_qual, "w"), indent=1, default=float)
print(json.load(open(arq_qual)))
del generate; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# 4) BERTimbau com os rótulos do v3 (condição L3, 3 sementes), mesmo protocolo da condição L
treino_llm = treino[["review_id", "texto_modelo"]].merge(rot_treino[rot_treino.valido], on="review_id")
print(f"treino com rótulos do LLM (v3): {len(treino_llm)} textos")
for seed in [13, 42, 87]:
    arq = SAIDA / f"pred_teste_bertimbau_L3_s{seed}.csv"
    if arq.exists():
        print(f"✓ L3 s{seed} já feito — pulando"); continue
    t0 = time.time()
    clf = EncoderAspectClassifier(TrainConfig(model_name=ENCODERS["bertimbau"], epochs=4, lr=3e-5,
                                              batch_size=32, max_len=128, seed=seed))
    ckpt = CKPT / f"L3_s{seed}.pt"
    clf.fit(entrada(treino_llm), treino_llm, entrada(val_dev), val_dev, save_path=ckpt)
    clf.load(ckpt)
    proba = clf.predict_proba(entrada(teste))
    p = pd.DataFrame({c: [POLARITIES[i] for i in proba[:, j].argmax(1)] for j, c in enumerate(LABEL_COLUMNS)})
    p.insert(0, "review_id", teste.review_id.values)
    p.to_csv(arq, index=False)
    json.dump({"cond": "L3", "seed": seed, "n_treino": len(treino_llm), "minutos": (time.time() - t0) / 60,
               "historico": clf.history}, open(SAIDA / f"log_bertimbau_L3_s{seed}.json", "w"), indent=1)
    print(f"→ L3 s{seed}: F1 de pares no teste {tuple_f1(teste, p):.3f} (L com v2: 0,794; H: 0,918)")
    ckpt.unlink(missing_ok=True); del clf; torch.cuda.empty_cache()

In [ ]:
# 5) Baixar os resultados (resultados_v3.zip) — também ficam no Drive
import shutil
shutil.make_archive("/content/resultados_v3", "zip", SAIDA)
if "google.colab" in sys.modules:
    from google.colab import files
    files.download("/content/resultados_v3.zip")